# Building Charts

In the explainer, we worked through the principles: position and length are the most accurately perceived encodings, chart type should follow from the relationship we want to communicate, and every visual element on a chart should earn its place. Now we get to put that into practice.

We are preparing charts for a climate policy briefing. The audience is policymakers who are not data specialists, so each chart needs one clear point. If the reader has to work out what they are looking at, the chart has failed.

By the end of this notebook we will be able to:

- Create **line charts**, **bar charts**, **scatter plots**, and **histograms** using matplotlib
- Add proper titles, axis labels, legends, and formatting to every chart
- Choose the right chart type for the data and the message
- Explain why pie charts are rarely the best choice

In [ ]:
%pip install -q pandas matplotlib seaborn

In [ ]:
%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

---

## Loading the data

The briefing draws on four datasets covering global temperature anomalies, energy generation by country, CO2 emissions, and renewable energy capacity. Have a look at what we are working with.

In [ ]:
temp = pd.read_csv("../data/global_temperature.csv")
energy = pd.read_csv("../data/energy_mix_by_country.csv")
co2 = pd.read_csv("../data/co2_emissions.csv")
renewables = pd.read_csv("../data/renewable_capacity.csv")

print(f"Temperature: {len(temp)} rows")
print(f"Energy mix:  {len(energy)} rows")
print(f"CO2:         {len(co2)} rows")
print(f"Renewables:  {len(renewables)} rows")

In [ ]:
temp.head()

---

## 1. Line charts with `plt.plot()`

We already know from the explainer that line charts are the standard choice for **trend** questions: "How has X changed over time?" They use position on a common scale for both axes, so the reader perceives differences accurately. The slope of the line communicates the rate of change.

Our first briefing chart: global temperature anomaly since 1850.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(temp["year"], temp["anomaly_c"], colour="#c0392b", linewidth=0.8)

ax.set_title("Global Temperature Anomaly (1850\u20132025)", fontsize=14, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Temperature anomaly (\u00b0C)")
ax.axhline(y=0, colour="grey", linestyle="--", linewidth=0.5)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

That cell will produce an error because matplotlib uses the American spelling `color`, not `colour`. This trips up almost everyone the first time. Replace `colour=` with `color=` in the matplotlib calls. (The rest of this notebook uses the matplotlib spelling.)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(temp["year"], temp["anomaly_c"], color="#c0392b", linewidth=0.8)

ax.set_title("Global Temperature Anomaly (1850\u20132025)", fontsize=14, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Temperature anomaly (\u00b0C)")
ax.axhline(y=0, color="grey", linestyle="--", linewidth=0.5)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

### What makes this chart work

Think back to the encoding hierarchy. This chart relies on the most accurate encoding — position on a common scale — for both year and temperature. A few deliberate design choices support the message:

- **One clear message**: temperatures have risen, especially since the 1970s.
- **Axis labels** tell the reader what the numbers mean.
- **Reference line** at zero gives context: anything above the line is warmer than the baseline.
- **Removed top and right spines** reduce clutter (Tufte would approve).

### Adding a rolling average

Year-to-year noise can distract from the trend. A **rolling average** smooths that out, giving the policymaker a cleaner signal to cite.

In [ ]:
temp["rolling_10yr"] = temp["anomaly_c"].rolling(window=10, center=True).mean()

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(temp["year"], temp["anomaly_c"], color="#e6b0aa", linewidth=0.6, label="Annual")
ax.plot(temp["year"], temp["rolling_10yr"], color="#c0392b", linewidth=2.0, label="10-year rolling average")

ax.set_title("Global Temperature Anomaly with 10-Year Rolling Average", fontsize=14, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Temperature anomaly (\u00b0C)")
ax.axhline(y=0, color="grey", linestyle="--", linewidth=0.5)
ax.legend(loc="upper left")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

The **legend** distinguishes the two lines. Placing it at `upper left` keeps it out of the data's path. Notice we are using a secondary encoding here: colour differentiates the raw series from the smoothed one, but the position encoding is still doing the heavy lifting.

### Comparing multiple lines

Line charts also handle **comparison** across groups when the x-axis is continuous. Here are CO2 emissions for five major economies. Keep the number of lines to five or fewer — beyond that, the chart becomes cluttered and the reader cannot track individual trajectories.

In [ ]:
focus_countries = ["China", "USA", "India", "Germany", "UK"]

fig, ax = plt.subplots(figsize=(10, 5))

for country in focus_countries:
    subset = co2[co2["country"] == country]
    ax.plot(subset["year"], subset["emissions_mt"], linewidth=1.5, label=country)

ax.set_title("CO\u2082 Emissions by Country (2000\u20132024)", fontsize=14, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Emissions (Mt CO\u2082)")
ax.legend()

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

---

## 2. Bar charts with `plt.bar()`

In the explainer, we saw that **comparison** questions ("Which is biggest? How does X compare to Y?") are best served by bar charts. They use position on a common scale, so the reader can judge differences with high precision.

For the briefing: total CO2 emissions by country in 2024.

In [ ]:
co2_2024 = co2[co2["year"] == 2024].sort_values("emissions_mt", ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))

ax.barh(co2_2024["country"], co2_2024["emissions_mt"], color="#2980b9")

ax.set_title("CO\u2082 Emissions by Country (2024)", fontsize=14, fontweight="bold")
ax.set_xlabel("Emissions (Mt CO\u2082)")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

**Horizontal bars** (`barh`) work better when category labels are long. Sorting from smallest to largest makes the ranking immediately visible — the reader's eye goes straight to the longest bar. This is one of those small design choices that separates a professional chart from a default one.

### Grouped bar chart

To compare the same categories across two time periods, place bars side by side. This lets the reader use position to compare within a year and across years simultaneously.

In [ ]:
import numpy as np

countries_ordered = ["UK", "Germany", "France", "Japan", "Australia",
                     "South Africa", "Brazil", "India", "USA", "China"]
co2_2005 = co2[co2["year"] == 2005].set_index("country").loc[countries_ordered]
co2_2024_g = co2[co2["year"] == 2024].set_index("country").loc[countries_ordered]

x = np.arange(len(countries_ordered))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width / 2, co2_2005["per_capita_t"], width, label="2005", color="#85c1e9")
ax.bar(x + width / 2, co2_2024_g["per_capita_t"], width, label="2024", color="#2471a3")

ax.set_title("Per-Capita CO\u2082 Emissions: 2005 vs 2024", fontsize=14, fontweight="bold")
ax.set_ylabel("Tonnes CO\u2082 per person")
ax.set_xticks(x)
ax.set_xticklabels(countries_ordered, rotation=45, ha="right")
ax.legend()

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

---

## 3. Scatter plots with `plt.scatter()`

A **scatter plot** shows the **relationship** between two numerical variables. Each point is one observation, and its position encodes both values simultaneously. This is the chart type we reach for when the question is "Is there a connection between X and Y?"

For the briefing: is there a relationship between a country's total emissions and its per-capita emissions?

In [ ]:
co2_latest = co2[co2["year"] == 2024].copy()

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(co2_latest["emissions_mt"], co2_latest["per_capita_t"],
           s=100, color="#27ae60", edgecolors="white", linewidth=0.8, zorder=5)

# Label each point with the country name
for _, row in co2_latest.iterrows():
    ax.annotate(row["country"],
                xy=(row["emissions_mt"], row["per_capita_t"]),
                xytext=(8, 0), textcoords="offset points",
                fontsize=9)

ax.set_title("Total vs Per-Capita CO\u2082 Emissions (2024)", fontsize=14, fontweight="bold")
ax.set_xlabel("Total emissions (Mt CO\u2082)")
ax.set_ylabel("Per-capita emissions (tonnes)")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

This chart makes an important policy point: high total emissions and high per-capita emissions are not the same thing. China has the highest total, but countries like Australia and the USA have much higher per-capita figures. A well-chosen chart type lets the data make the argument.

### When to use scatter plots

Use scatter plots when we want to show the **relationship** between two measured quantities. They are not useful when one axis is categorical (use a bar chart) or when there are thousands of overlapping points (consider a hexbin or density plot instead).

---

## 4. Histograms with `plt.hist()`

The fourth relationship type from the explainer is **distribution**: "How is the data spread?" A **histogram** groups continuous data into bins and counts how many observations fall in each bin. It reveals the shape of the data in a way no summary statistic can.

For the briefing: what is the distribution of per-capita emissions across all country-years?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(co2["per_capita_t"], bins=25, color="#8e44ad", edgecolor="white", linewidth=0.5)

ax.set_title("Distribution of Per-Capita CO\u2082 Emissions (All Countries, 2000\u20132024)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Per-capita emissions (tonnes CO\u2082)")
ax.set_ylabel("Number of country-year observations")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

Most country-years have low per-capita emissions, with a long right tail from high-emitting countries. That skewed shape is exactly the kind of insight a histogram reveals instantly. A policymaker can see at a glance that the global distribution is heavily concentrated at the low end, with a handful of outliers pulling the average up.

### Choosing the number of bins

Too few bins hide the shape of the distribution. Too many bins create noise. Start with 20-30 bins and adjust until the shape is clear without being jagged.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, n_bins in zip(axes, [5, 25, 80]):
    ax.hist(co2["per_capita_t"], bins=n_bins, color="#8e44ad", edgecolor="white", linewidth=0.5)
    ax.set_title(f"{n_bins} bins")
    ax.set_xlabel("Per-capita (t)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle("Effect of Bin Count on Histogram Shape", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---

## 5. Why not pie charts?

We saw in the explainer that angle sits well below position and length on Cleveland and McGill's accuracy ranking. Pie charts encode proportions as angles, and humans are poor at comparing them. When there are more than three or four slices, the chart becomes genuinely difficult to read.

Compare these two views of the UK's energy mix in 2024:

In [ ]:
uk_2024 = energy[(energy["country"] == "UK") & (energy["year"] == 2024)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
ax1.pie(uk_2024["generation_twh"], labels=uk_2024["source"], autopct="%1.0f%%",
        startangle=140)
ax1.set_title("UK Energy Mix 2024 (pie)")

# Bar chart of the same data
sorted_uk = uk_2024.sort_values("generation_twh", ascending=True)
ax2.barh(sorted_uk["source"], sorted_uk["generation_twh"], color="#2980b9")
ax2.set_title("UK Energy Mix 2024 (bar)")
ax2.set_xlabel("Generation (TWh)")
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

The bar chart makes it straightforward to compare the exact values. The pie chart forces us to estimate angles. For a briefing where precision matters, the bar chart wins every time.

**When a pie chart might be acceptable**: when there are only two or three categories and the goal is to show that one dominates. Even then, a stacked bar is usually clearer. If we find ourselves reaching for a pie chart, it is worth pausing to ask whether position or length could do the job instead.

---

## Summary of chart types

We have now built one chart for each of the four relationship types from the explainer. Here is the mapping:

| Chart type | Relationship | Example |
|---|---|---|
| Line (`plt.plot`) | Trend over time | Temperature, emissions over years |
| Bar (`plt.bar`, `plt.barh`) | Comparison across categories | Emissions by country |
| Scatter (`plt.scatter`) | Relationship between two variables | Total vs per-capita emissions |
| Histogram (`plt.hist`) | Distribution of values | Spread of per-capita emissions |

---

## Exercises

Time to build your own briefing charts. Each exercise asks you to choose appropriate formatting and create a chart that a policymaker could read without explanation.

### Exercise 1: Renewable capacity over time

Create a line chart showing solar capacity (GW) from 2010 to 2024 for China, USA, India, and Germany. Include a title, axis labels, and a legend. Remove the top and right spines.

In [ ]:
# Your code here


### Exercise 2: Energy sources bar chart

Create a horizontal bar chart showing generation (TWh) by energy source for Germany in 2024. Sort the bars from largest to smallest. Add a title and axis labels.

In [ ]:
# Your code here


### Exercise 3: Wind vs solar scatter

Using the `renewables` dataframe for the year 2024, create a scatter plot with `solar_gw` on the x-axis and `wind_gw` on the y-axis. Label each point with the country name using `ax.annotate()`. Give the chart a title and axis labels.

In [ ]:
# Your code here


### Exercise 4: Temperature anomaly histogram

Create a histogram of the global temperature anomaly values. Use 30 bins. Add a vertical line at the mean anomaly using `ax.axvline()`. Include a title and axis labels.

In [ ]:
# Your code here


---

## Summary

We have gone from the principles in the explainer to working charts in matplotlib. Specifically, we:

- Built **line charts** to show temperature and emissions trends over time
- Used **bar charts** to compare emissions across countries
- Created **scatter plots** to reveal relationships between two variables
- Made **histograms** to show how per-capita emissions are distributed
- Learned why **pie charts** are usually the wrong choice for a professional briefing
- Applied consistent formatting: titles, axis labels, legends, and clean spines

Every chart we built maps directly to one of the four relationship types (trend, comparison, relationship, distribution), and every formatting choice follows from the encoding principles we already know. In the next notebook, we will take charts that break those principles and fix them.